# 📊 Exploratory Data Analysis — Credit Card Default
> **Dataset:** UCI Credit Card Default (Taiwan, 2005)  
> **Objetivo:** Entender o perfil dos clientes inadimplentes, identificar padrões e preparar o terreno para modelagem preditiva.

---

## 📋 Índice
1. [Setup & Carregamento](#1-setup)
2. [Visão Geral do Dataset](#2-overview)
3. [Qualidade dos Dados](#3-qualidade)
4. [Variável Alvo — `default`](#4-target)
5. [Perfil Demográfico](#5-demografico)
6. [Limite de Crédito](#6-limite)
7. [Comportamento de Pagamento](#7-pagamento)
8. [Faturas ao Longo do Tempo](#8-faturas)
9. [Correlações](#9-correlacoes)
10. [Conclusões para Modelagem](#10-conclusoes)


## 1. Setup & Carregamento <a id="1-setup"></a>
Configuração de estilo e importação das funções de pré-processamento.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import sys
sys.path.append('../src')

from preprocessing import rename_columns, transform_variables, clean_data, process_status

# ── Tema global ─────────────────────────────────────────────────────────────
PALETTE   = ['#2ecc71', '#e74c3c']          # verde = não default, vermelho = default
PALETTE_C = ['#e74c3c', '#c0392b', '#922b21']  # escala sequencial para default
BG        = '#f9f9f9'

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({
    'figure.facecolor': BG,
    'axes.facecolor':   BG,
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'figure.dpi':       120,
})
#Usar para o ticker do matplotlib (funcformatter)
#Se o valor for acima de 1000, será dividido e formatado com k após o valor
def fmt_currency(x, pos):
    """Formata eixo em NT$ milhares."""
    return f'NT${x/1_000:.0f}k' if abs(x) >= 1000 else f'NT${x:.0f}'

print('Setup concluído ✓')


In [ ]:
df_raw = pd.read_csv('../data/credit.csv')
#Load do dataset com as transformações das colunas previamente analisadas com drop do id
df = (df_raw
      .pipe(rename_columns)
      .drop(df_raw.index[0])
      .pipe(transform_variables)
      .pipe(clean_data)
      .pipe(process_status))

print(f'Shape após pré-processamento: {df.shape}')
df.head()


## 2. Visão Geral do Dataset <a id="2-overview"></a>

### Dicionário de variáveis

| Grupo | Variável | Descrição |
|-------|----------|-----------|
| Identificação | `id` | ID único do cliente |
| Perfil | `limit_bal` | Limite de crédito concedido (NT$) |
| Perfil | `sex` | Sexo (1=M, 2=F) |
| Perfil | `education` | Escolaridade (1=pós-grad, 2=universidade, 3=ensino médio, 4=outros) |
| Perfil | `marriage` | Estado civil (1=casado, 2=solteiro, 3=outros) |
| Perfil | `age` | Idade (anos) |
| Status de pgto | `pay_1`…`pay_6` | Meses de atraso em set–abr/2005 (0=em dia, 1–8=meses de atraso) |
| Faturas | `bill_amt1`…`bill_amt6` | Valor da fatura em set–abr/2005 (NT$) |
| Pagamentos | `pay_amt1`…`pay_amt6` | Valor pago em set–abr/2005 (NT$) |
| **Target** | `default` | **1 = inadimplente no mês seguinte, 0 = não** |

> ⚠️ `pay_*` foi recodificado: valores `-2`, `-1` e `0` tratados como `0` (pagamento em dia).  
> `education` 0/5/6 agrupados em `4` (outros). `marriage` 0 mapeado para `3` (outros).


In [ ]:
df.info()


In [ ]:
df.describe().round(1)


## 3. Qualidade dos Dados <a id="3-qualidade"></a>
Verificação de valores nulos e duplicatas.


In [ ]:
# Nulos
nulls = df.isnull().sum().to_frame('nulos')
nulls['%'] = (nulls['nulos'] / len(df) * 100).round(2)
print("=== Valores Nulos ===")
print(nulls[nulls['nulos'] > 0] if nulls['nulos'].sum() > 0 else "Nenhum valor nulo encontrado ✓")

# Duplicatas
dupes = df.duplicated().sum()
print(f"\n=== Duplicatas: {dupes} ===")


## 4. Variável Alvo — `default` <a id="4-target"></a>

O dataset é **desbalanceado**: a classe minoritária (inadimplentes) representa ~22% dos registros.  
Isso impacta diretamente a escolha de métricas (PR-AUC > ROC-AUC) e estratégias de treino (SMOTE, class_weight).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ── Gráfico 1: contagem absoluta ────────────────────────────────────────────
counts = df['default'].value_counts().sort_index()
labels = ['Adimplente\n(0)', 'Inadimplente\n(1)']
bars   = axes[0].bar(labels, counts, color=PALETTE, width=0.5, edgecolor='white', linewidth=1.5)

for bar, count in zip(bars, counts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                 f'{count:,}', ha='center', va='bottom', fontweight='bold')

axes[0].set_title('Distribuição Absoluta', fontsize=13, fontweight='bold', pad=12)
axes[0].set_ylabel('Contagem de Clientes')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# ── Gráfico 2: proporção em pizza ───────────────────────────────────────────
pct    = counts / counts.sum() * 100
wedges, texts, autotexts = axes[1].pie(
    pct, labels=[f'Adimplente\n{pct[0]:.1f}%', f'Inadimplente\n{pct[1]:.1f}%'],
    colors=PALETTE, autopct='%1.1f%%', startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=2),
    textprops=dict(fontsize=11)
)
for at in autotexts:
    at.set_visible(False)  # evita texto duplo

axes[1].set_title('Proporção por Classe', fontsize=13, fontweight='bold', pad=12)

fig.suptitle('Desbalanceamento da Variável Alvo', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../plots/target_distribution.png', bbox_inches='tight')
plt.show()

print(f"Taxa de inadimplência: {pct[1]:.2f}%")
print("→ Dataset desbalanceado. Usar SMOTE ou class_weight na modelagem.")


## 5. Perfil Demográfico <a id="5-demografico"></a>

Analisamos as variáveis categóricas **segmentadas por default** para identificar grupos de maior risco.


In [ ]:
# ── Mapeamentos legíveis ────────────────────────────────────────────────────
df_plot = df.copy()
df_plot['sex_label']       = df_plot['sex'].map({1:'Masculino', 2:'Feminino'})
df_plot['education_label'] = df_plot['education'].map({
    1:'Pós-Grad', 2:'Universidade', 3:'Ens. Médio', 4:'Outros'})
df_plot['marriage_label']  = df_plot['marriage'].map({
    1:'Casado', 2:'Solteiro', 3:'Outros'})
df_plot['default_label']   = df_plot['default'].map({0:'Adimplente', 1:'Inadimplente'})

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, col, title, order in zip(
    axes,
    ['sex_label', 'education_label', 'marriage_label'],
    ['Sexo', 'Escolaridade', 'Estado Civil'],
    [['Masculino','Feminino'],
     ['Pós-Grad','Universidade','Ens. Médio','Outros'],
     ['Casado','Solteiro','Outros']]
):
    # Taxa de default por categoria
    rates = (df_plot.groupby(col)['default']
             .mean()
             .reindex(order)
             .reset_index())
    rates.columns = [col, 'taxa_default']

    bars = ax.bar(rates[col], rates['taxa_default'] * 100,
                  color='#e74c3c', alpha=0.75, edgecolor='white', linewidth=1.5)

    for bar, val in zip(bars, rates['taxa_default']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{val*100:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

    ax.set_title(f'Taxa de Default por {title}', fontsize=12, fontweight='bold', pad=10)
    ax.set_ylabel('Taxa de Default (%)' if ax == axes[0] else '')
    ax.set_ylim(0, rates['taxa_default'].max() * 100 * 1.25)
    ax.tick_params(axis='x', rotation=15)

    # Linha de referência = média global
    ax.axhline(df['default'].mean() * 100, color='gray', linestyle='--',
               linewidth=1, label=f'Média geral ({df["default"].mean()*100:.1f}%)')
    ax.legend(fontsize=8)

fig.suptitle('Taxa de Inadimplência por Perfil Demográfico', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../plots/demographic_default_rate.png', bbox_inches='tight')
plt.show()


**Leitura:** A linha tracejada é a taxa média global (~22%). Grupos acima dela são proporcionalmente mais arriscados.  
Nenhuma variável demográfica isolada separa bem os grupos — confirma que o modelo precisa de combinações de features.


In [ ]:
# ── Distribuição de Idade por Default ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Histograma sobreposto
for val, label, color in zip([0, 1], ['Adimplente', 'Inadimplente'], PALETTE):
    subset = df[df['default'] == val]['age']
    axes[0].hist(subset, bins=30, alpha=0.6, color=color, label=label, edgecolor='white')

axes[0].set_title('Distribuição de Idade por Default', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Idade')
axes[0].set_ylabel('Contagem')
axes[0].legend()

# Boxplot lado a lado
df_plot_age = df_plot[['age', 'default_label']].copy()
sns.boxplot(x='default_label', y='age', data=df_plot_age,
            palette={'Adimplente': PALETTE[0], 'Inadimplente': PALETTE[1]},
            width=0.4, ax=axes[1])
axes[1].set_title('Boxplot de Idade por Default', fontsize=12, fontweight='bold')
axes[1].set_xlabel('')
axes[1].set_ylabel('Idade')

# Estatísticas
for val, label in zip([0, 1], ['Adimplente', 'Inadimplente']):
    m = df[df['default'] == val]['age'].median()
    print(f'{label}: mediana de idade = {m:.0f} anos')

plt.tight_layout()
plt.savefig('../plots/age_distribution.png', bbox_inches='tight')
plt.show()


## 6. Limite de Crédito <a id="6-limite"></a>

`limit_bal` é o limite total concedido ao cliente. Esperamos que clientes com **limites menores** tenham maior propensão ao default, pois limites são concedidos com base na capacidade de pagamento.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# ── Histograma com KDE por grupo ─────────────────────────────────────────────
for val, label, color in zip([0, 1], ['Adimplente', 'Inadimplente'], PALETTE):
    subset = df[df['default'] == val]['limit_bal']
    axes[0].hist(subset, bins=60, alpha=0.5, color=color, label=label,
                 density=True, edgecolor='none')

# KDE
for val, label, color in zip([0, 1], ['Adimplente', 'Inadimplente'], PALETTE):
    subset = df[df['default'] == val]['limit_bal']
    from scipy.stats import gaussian_kde
    kde = gaussian_kde(subset, bw_method=0.15)
    xs  = np.linspace(subset.min(), subset.max(), 300)
    axes[0].plot(xs, kde(xs), color=color, linewidth=2)

axes[0].set_title('Distribuição do Limite de Crédito', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Limite de Crédito (NT$)')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(fmt_currency))
axes[0].tick_params(axis='x', rotation=30)
axes[0].legend()

# ── Boxplot comparativo ───────────────────────────────────────────────────────
sns.boxplot(x='default_label', y='limit_bal', data=df_plot,
            palette={'Adimplente': PALETTE[0], 'Inadimplente': PALETTE[1]},
            width=0.4, showfliers=False, ax=axes[1])
axes[1].set_title('Limite de Crédito vs Default\n(sem outliers extremos)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('')
axes[1].set_ylabel('Limite de Crédito (NT$)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_currency))

# Medianas
for val, label in zip([0, 1], ['Adimplente', 'Inadimplente']):
    m = df[df['default'] == val]['limit_bal'].median()
    print(f'{label}: mediana do limite = NT${m:,.0f}')

plt.tight_layout()
plt.savefig('../plots/limit_bal_analysis.png', bbox_inches='tight')
plt.show()


**Leitura:** Clientes inadimplentes tendem a ter **limites de crédito significativamente menores**.  
Isso sugere que o banco já percebia maior risco nesse grupo no momento da concessão.  
`limit_bal` será uma feature importante no modelo.


## 7. Comportamento de Pagamento <a id="7-pagamento"></a>

As variáveis `pay_1` a `pay_6` indicam o número de meses de atraso em cada mês de set–abr/2005.  
São as features mais preditivas — atraso recente é forte sinal de risco.


In [ ]:
pay_cols = ['pay_1', 'pay_2', 'pay_3', 'pay_4', 'pay_5', 'pay_6']
months   = ['Set/05', 'Ago/05', 'Jul/05', 'Jun/05', 'Mai/05', 'Abr/05']

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for ax, col, month in zip(axes, pay_cols, months):
    # Taxa de default para cada nível de atraso
    rates = (df.groupby(col)['default']
               .agg(['mean', 'count'])
               .reset_index())
    rates.columns = ['atraso', 'taxa_default', 'n']

    bars = ax.bar(rates['atraso'].astype(str), rates['taxa_default'] * 100,
                  color='#e74c3c', alpha=0.75, edgecolor='white', linewidth=1.2)

    # Número de clientes em cada bin
    for bar, row in zip(bars, rates.itertuples()):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'n={row.n:,}', ha='center', va='bottom', fontsize=7, color='gray')

    ax.axhline(df['default'].mean() * 100, color='gray', linestyle='--', linewidth=1)
    ax.set_title(f'{col.upper()} — {month}', fontsize=11, fontweight='bold')
    ax.set_xlabel('Meses de atraso')
    ax.set_ylabel('Taxa de Default (%)')
    ax.set_ylim(0, 105)

fig.suptitle('Taxa de Default por Status de Pagamento (mês a mês)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../plots/pay_status_default_rate.png', bbox_inches='tight')
plt.show()


**Leitura:** 
- Clientes **em dia (0)** têm taxa de default próxima da média ou abaixo.  
- A partir de **1 mês de atraso**, a taxa de default dispara para 50–80%.  
- `pay_1` (mês mais recente) é a feature mais "quente" — atraso recente prediz default melhor que atraso antigo.  
- A distribuição assimétrica (maioria em 0) confirma o desbalanceamento.


In [ ]:
# ── Heatmap: % de defaults por combinação de atraso ────────────────────────
# Agrupa atraso em categorias para facilitar leitura
df_heat = df.copy()
for col in pay_cols:
    df_heat[col] = df_heat[col].clip(0, 3).map({0:'0 (em dia)', 1:'1 mês', 2:'2 meses', 3:'3+ meses'})

pivot = df_heat.pivot_table(index='pay_1', columns='pay_2',
                             values='default', aggfunc='mean') * 100

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='Reds',
            linewidths=0.5, linecolor='white',
            cbar_kws={'label': 'Taxa de Default (%)'}, ax=ax)
ax.set_title('Taxa de Default: Atraso em Set/05 (pay_1) × Ago/05 (pay_2)',
             fontsize=12, fontweight='bold', pad=12)
ax.set_xlabel('pay_2 — Ago/05 (meses de atraso)')
ax.set_ylabel('pay_1 — Set/05 (meses de atraso)')

plt.tight_layout()
plt.savefig('../plots/pay_heatmap.png', bbox_inches='tight')
plt.show()


## 8. Faturas ao Longo do Tempo <a id="8-faturas"></a>

Analisamos como o saldo devedor evolui nos 6 meses observados.  
Clientes inadimplentes podem ter faturas crescentes (consumindo sem pagar) ou decrescentes (pararam de usar antes do default).


In [ ]:
bill_cols    = ['bill_amt1', 'bill_amt2', 'bill_amt3', 'bill_amt4', 'bill_amt5', 'bill_amt6']
pay_amt_cols = ['pay_amt1',  'pay_amt2',  'pay_amt3',  'pay_amt4',  'pay_amt5',  'pay_amt6']
months_rev   = ['Set/05', 'Ago/05', 'Jul/05', 'Jun/05', 'Mai/05', 'Abr/05']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for val, label, color in zip([0, 1], ['Adimplente', 'Inadimplente'], PALETTE):
    subset = df[df['default'] == val]
    
    # Mediana das faturas ao longo do tempo
    bill_medians = subset[bill_cols].median()
    pay_medians  = subset[pay_amt_cols].median()
    
    axes[0].plot(months_rev, bill_medians, marker='o', color=color, label=label, linewidth=2.5)
    axes[1].plot(months_rev, pay_medians,  marker='o', color=color, label=label, linewidth=2.5)

axes[0].set_title('Mediana da Fatura (bill_amt)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('NT$ (mediana)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_currency))
axes[0].legend()
axes[0].tick_params(axis='x', rotation=30)

axes[1].set_title('Mediana do Pagamento (pay_amt)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('NT$ (mediana)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_currency))
axes[1].legend()
axes[1].tick_params(axis='x', rotation=30)

fig.suptitle('Evolução Temporal: Faturas e Pagamentos por Grupo',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../plots/temporal_evolution.png', bbox_inches='tight')
plt.show()


**Leitura:**
- Inadimplentes tendem a ter **faturas similares ou maiores** que adimplentes, mas **pagam valores muito menores**.
- A diferença no `pay_amt` (pagamento) é mais gritante que no `bill_amt` (fatura).
- Isso sugere que criar a feature `pay_ratio = pay_amt / bill_amt` (razão pagamento/fatura) captura bem o comportamento de risco.


In [ ]:
# ── Scatter: fatura mais recente vs pagamento mais recente ──────────────────
fig, ax = plt.subplots(figsize=(9, 5))

for val, label, color in zip([0, 1], ['Adimplente', 'Inadimplente'], PALETTE):
    subset = df[df['default'] == val].sample(500, random_state=42)
    ax.scatter(subset['bill_amt1'], subset['pay_amt1'],
               alpha=0.25, s=15, color=color, label=label)

ax.set_title('Fatura Set/05 × Pagamento Set/05', fontsize=12, fontweight='bold')
ax.set_xlabel('Fatura (bill_amt1) — NT$')
ax.set_ylabel('Pagamento Realizado (pay_amt1) — NT$')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(fmt_currency))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_currency))
ax.legend(markerscale=2)

# Linha de referência: pagou 100% da fatura
max_val = min(df['bill_amt1'].quantile(0.95), df['pay_amt1'].quantile(0.99))
ax.plot([0, max_val], [0, max_val], 'k--', linewidth=1, alpha=0.4, label='Pagou 100%')
ax.set_xlim(0, max_val)
ax.set_ylim(0, max_val * 0.5)

plt.tight_layout()
plt.savefig('../plots/bill_vs_payment_scatter.png', bbox_inches='tight')
plt.show()


## 9. Correlações <a id="9-correlacoes"></a>

Analisamos duas perspectivas:
1. **Correlação com o target** — quais features individualmente têm mais poder preditivo
2. **Correlação entre features** — identificar multicolinearidade (features redundantes)


In [ ]:
df_num = df.drop(columns=['id'], errors='ignore')
corr   = df_num.corr()

# ── 1. Correlação com o target (barras horizontais) ─────────────────────────
target_corr = (corr['default']
               .drop('default')
               .sort_values(key=abs, ascending=False))

fig, ax = plt.subplots(figsize=(8, 10))
colors  = ['#e74c3c' if v > 0 else '#2ecc71' for v in target_corr]
bars    = ax.barh(target_corr.index[::-1], target_corr.values[::-1],
                  color=colors[::-1], edgecolor='white', height=0.7)

ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Correlação de cada Feature com `default`',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Correlação de Pearson')

# Valores nas barras
for bar, val in zip(bars, target_corr.values[::-1]):
    x_pos = bar.get_width() + 0.003 if val >= 0 else bar.get_width() - 0.003
    ax.text(x_pos, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', ha='left' if val >= 0 else 'right', fontsize=8)

plt.tight_layout()
plt.savefig('../plots/target_correlation.png', bbox_inches='tight')
plt.show()

print("Top 5 features mais correlacionadas com default:")
print(target_corr.head())


In [ ]:
# ── 2. Heatmap de correlação entre features (apenas as mais relevantes) ────
top_features = target_corr.head(12).index.tolist() + ['default']
corr_top     = df_num[top_features].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask    = np.triu(np.ones_like(corr_top, dtype=bool))  # mostra só metade inferior

sns.heatmap(corr_top, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            linewidths=0.5, linecolor='white',
            cbar_kws={'label': 'Correlação', 'shrink': 0.7}, ax=ax)

ax.set_title('Correlação entre as Features Mais Relevantes',
             fontsize=13, fontweight='bold', pad=12)

plt.tight_layout()
plt.savefig('../plots/correlation_heatmap_top.png', bbox_inches='tight')
plt.show()


**Leitura:**
- As variáveis `pay_1` a `pay_6` são as mais correlacionadas com `default` — **comportamento de atraso é o melhor sinal**.
- `bill_amt1`–`bill_amt6` são altamente correlacionadas entre si (multicolinearidade) — considerar criar uma feature agregada como `avg_bill`.
- `limit_bal` tem correlação negativa com default: maior limite → menor risco.


## 10. Conclusões para Modelagem <a id="10-conclusoes"></a>

### 🔍 Findings principais

| Tema | Achado | Impacto na Modelagem |
|------|--------|----------------------|
| **Desbalanceamento** | ~22% de defaults | Usar SMOTE ou `class_weight`; métricas: PR-AUC, Recall |
| **Variáveis de atraso** | `pay_1`–`pay_6` mais correlacionadas com target | Feature importance alta; considerar `n_delays` e `max_delay` |
| **Limite de crédito** | Inadimplentes têm limites menores | `limit_bal` feature relevante; criar `util_ratio` |
| **Faturas** | Alta multicolinearidade entre `bill_amt*` | Agregar em `avg_bill`; criar `pay_ratio` |
| **Perfil demográfico** | Sexo, escolaridade e estado civil têm impacto pequeno | Usar, mas não são os principais preditores |

### ✅ Features de engenharia recomendadas

```python
# Em src/features.py
df['util_ratio']  = df['avg_bill'] / (df['limit_bal'] + 1)  # % do limite usado
df['pay_ratio_1'] = df['pay_amt1'] / (df['bill_amt2'].abs() + 1)  # pagou quanto da fatura?
df['n_delays']    = (df[pay_cols] > 0).sum(axis=1)  # quantos meses com atraso
df['max_delay']   = df[pay_cols].max(axis=1)  # pior mês de atraso
```

### 📐 Métricas de avaliação

| Métrica | Por quê usar |
|---------|-------------|
| **PR-AUC** | Melhor para datasets desbalanceados |
| **Recall** | Queremos capturar o máximo de inadimplentes reais |
| **Precision** | Evitar falsos alarmes excessivos |
| **ROC-AUC** | Referência geral de separabilidade |

### 🚀 Próximos passos

1. `src/features.py` — implementar feature engineering  
2. `notebooks/modeling.ipynb` — split estratificado, baseline logístico, XGBoost + SMOTE  
3. Threshold tuning pela curva Precision-Recall  
4. SHAP para explicabilidade do modelo final  
